# Konfoundy na embeddingach — demo do prześledzenia

Notebook do **samodzielnego eksperymentowania** z detekcją dziurki (hole punch) i matematyką jej neutralizacji na embeddingach.

## Co robi
1. **Detekcja klasyczna** — wykrywa dziurkę bez ML (progowanie + analiza konturów)
2. **Inpainting** — tworzy wersję bez dziurki (ten sam obraz, dwie wersje)
3. **Matematyka embeddingów** — kierunek dziurki, projekcja ortogonalna, weryfikacja

## Jak używać
Załaduj własne zdjęcia do Colab (panel plików po lewej, lub komórka upload poniżej) i uruchamiaj po kolei. Działa z CLIP (jeśli się pobierze) lub prostym embedderem (fallback) — matematyka identyczna.


## 0. Setup

In [ ]:
!pip install -q opencv-python-headless open_clip_torch 2>/dev/null
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
np.set_printoptions(precision=4, suppress=True)
print('Gotowe')


## 1. Załaduj zdjęcia

Dwie opcje:
- **A**: wgraj pliki przez panel plików Colab (ikona folderu po lewej) do `/content/`
- **B**: użyj komórki upload poniżej

Ustaw `IMG_DIR` na folder z Twoimi zdjęciami (domyślnie `/content`).


In [ ]:
# Opcja B: upload bezpośrednio
from google.colab import files
print('Wybierz zdjęcia do wgrania (lub pomiń i użyj panelu plików):')
try:
    uploaded = files.upload()
    print(f'Wgrano: {list(uploaded.keys())}')
except Exception:
    print('Pominięto upload — użyj panelu plików')


In [ ]:
IMG_DIR = Path('/content')   # gdzie są Twoje zdjęcia

# Znajdź wszystkie obrazy
exts = ('.jpg', '.jpeg', '.png')
image_files = [f for f in IMG_DIR.iterdir() if f.suffix.lower() in exts]
image_files.sort()
print(f'Znaleziono {len(image_files)} obrazów:')
for f in image_files:
    print(f'  {f.name}')
if not image_files:
    print('⚠ Brak obrazów — wgraj pliki do /content')


## 2. Detektor dziurki (klasyczne CV)

In [ ]:
def detect_hole(img_bgr):
    """Wykrywa hole punch: bardzo ciemny, wypełniony, okrągły obszar WEWNĄTRZ kadru."""
    h, w = img_bgr.shape[:2]
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # 1. Progowanie — bardzo ciemne piksele
    _, dark = cv2.threshold(gray, 30, 255, cv2.THRESH_BINARY_INV)

    # 2. Usuń ramkę filmu (margines 12%)
    m_y, m_x = int(0.12*h), int(0.12*w)
    center = np.zeros_like(dark); center[m_y:h-m_y, m_x:w-m_x] = 1
    dark = dark * center

    # 3. Domknięcie morfologiczne
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
    dark = cv2.morphologyEx(dark, cv2.MORPH_CLOSE, k)

    # 4. Analiza konturów
    contours, _ = cv2.findContours(dark, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    best = None
    for c in contours:
        area = cv2.contourArea(c)
        if area < 80: continue
        perim = cv2.arcLength(c, True)
        if perim == 0: continue
        circularity = 4*np.pi*area / (perim*perim)
        (cx, cy), radius = cv2.minEnclosingCircle(c)
        fill = area / (np.pi*radius*radius + 1e-6)
        if circularity > 0.6 and fill > 0.55 and 6 < radius < 0.25*min(h,w):
            score = circularity * fill
            if best is None or score > best['score']:
                best = dict(cx=cx, cy=cy, radius=radius, area=area,
                            circularity=circularity, fill=fill, score=score)
    return best

# Uruchom detekcję na wszystkich
results = {}
for f in image_files:
    img = cv2.imread(str(f))
    hole = detect_hole(img)
    results[f.name] = hole
    if hole:
        print(f'{f.name}: DZIURKA ({hole["cx"]:.0f},{hole["cy"]:.0f}) r={hole["radius"]:.0f}px '
              f'circ={hole["circularity"]:.2f} fill={hole["fill"]:.2f}')
    else:
        print(f'{f.name}: brak dziurki')


## 3. Wizualizacja: detekcja + maska + inpainting

In [ ]:
def make_clean(img_bgr, hole):
    """Tworzy maskę i wersję bez dziurki (inpainting)."""
    mask = np.zeros(img_bgr.shape[:2], np.uint8)
    if hole:
        cv2.circle(mask, (int(hole['cx']), int(hole['cy'])), int(hole['radius'])+4, 255, -1)
    clean = cv2.inpaint(img_bgr, mask, 5, cv2.INPAINT_TELEA) if hole else img_bgr.copy()
    return mask, clean

# Pokaż obrazy z dziurką
with_hole = [f for f in image_files if results[f.name]]
n = min(len(with_hole), 4)
if n > 0:
    fig, axes = plt.subplots(n, 3, figsize=(12, 4*n))
    if n == 1: axes = axes.reshape(1, 3)
    for i in range(n):
        f = with_hole[i]; hole = results[f.name]
        img = cv2.imread(str(f)); rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask, clean = make_clean(img, hole)
        vis = rgb.copy()
        cv2.circle(vis, (int(hole['cx']), int(hole['cy'])), int(hole['radius'])+3, (255,0,0), 3)
        axes[i,0].imshow(vis); axes[i,0].set_title(f'{f.name}\ndetekcja', fontsize=9); axes[i,0].axis('off')
        axes[i,1].imshow(mask, cmap='gray'); axes[i,1].set_title('maska', fontsize=9); axes[i,1].axis('off')
        axes[i,2].imshow(cv2.cvtColor(clean, cv2.COLOR_BGR2RGB)); axes[i,2].set_title('po inpaintingu', fontsize=9); axes[i,2].axis('off')
    plt.tight_layout(); plt.show()
else:
    print('Brak obrazów z dziurką do wizualizacji')


## 4. Embedder — CLIP lub fallback

Próbujemy załadować CLIP. Jeśli się nie uda (brak sieci), używamy prostego embeddera siatkowego — **matematyka projekcji jest identyczna**, tylko CLIP lepiej "widzi" dziurkę jako kształt.


In [ ]:
EMBEDDER = None
try:
    import torch, open_clip
    model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
    model.eval()
    from PIL import Image
    def clip_embed(img_bgr):
        rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        pil = Image.fromarray(rgb)
        with torch.no_grad():
            e = model.encode_image(preprocess(pil).unsqueeze(0))
            e = e / e.norm(dim=-1, keepdim=True)
        return e[0].cpu().numpy()
    EMBEDDER = clip_embed
    print('✅ CLIP ViT-B/32 załadowany (embedding dim 512)')
except Exception as ex:
    print(f'⚠ CLIP niedostępny ({type(ex).__name__}), używam fallback embeddera')
    def grid_embed(img_bgr, dim=64):
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32)/255.0
        gray = cv2.resize(gray, (32,32))
        feats = [gray[gy*4:(gy+1)*4, gx*4:(gx+1)*4].mean() for gy in range(8) for gx in range(8)]
        e = np.array(feats, np.float32)
        return e / (np.linalg.norm(e)+1e-8)
    EMBEDDER = grid_embed
    print('   (fallback: embedding siatkowy 64-wym)')


## 5. KROK 1 — jak bardzo dziurka zmienia embedding

In [ ]:
def cos(a, b):
    return float(np.dot(a,b) / (np.linalg.norm(a)*np.linalg.norm(b) + 1e-8))

# Buduj pary (orig z dziurką, clean bez) dla obrazów z dziurką
E_orig, E_clean, names = [], [], []
for f in with_hole:
    img = cv2.imread(str(f)); hole = results[f.name]
    _, clean = make_clean(img, hole)
    E_orig.append(EMBEDDER(img))
    E_clean.append(EMBEDDER(clean))
    names.append(f.name)
E_orig = np.array(E_orig); E_clean = np.array(E_clean)

print('Jak bardzo dziurka zmienia embedding:\n')
for i, nm in enumerate(names):
    d = np.linalg.norm(E_orig[i]-E_clean[i])
    print(f'  {nm}:')
    print(f'     ||e_dziurka - e_clean|| = {d:.4f}')
    print(f'     cos(e_dziurka, e_clean) = {cos(E_orig[i], E_clean[i]):.4f}')


## 6. KROK 2 — kierunek dziurki v_hole

In [ ]:
# v_hole = mean(orig) - mean(clean)
v_hole = E_orig.mean(0) - E_clean.mean(0)
v_hole_norm = v_hole / (np.linalg.norm(v_hole) + 1e-8)

print(f'||v_hole|| = {np.linalg.norm(v_hole):.4f}')
print(f'Wymiar embeddingu: {len(v_hole)}\n')
print('Najsilniejsze wymiary v_hole (gdzie dziurka najmocniej zmienia embedding):')
top = np.argsort(-np.abs(v_hole))[:8]
for idx in top:
    print(f'   wymiar {idx:4d}: {v_hole[idx]:+.4f}')
print('\n(Dla embeddera siatkowego te wymiary odpowiadają komórkom w centrum = miejsce dziurki)')


## 6b. Struktura zmiany — lokalna czy rozproszona?

Czy dziurka zmienia KAŻDY element wektora, czy konkretne? I czy te same wymiary dla różnych zdjęć (spójny kierunek)? To decyduje, czy projekcja zadziała.


In [ ]:
# Wektory różnicy per zdjęcie
diffs = E_orig - E_clean   # (n_par, dim)

print('Ile wymiarów faktycznie się zmienia (na zdjęcie):')
for i, nm in enumerate(names):
    d = diffs[i]
    thr = 0.1 * np.abs(d).max()
    n_changed = int((np.abs(d) > thr).sum())
    top5_frac = np.sort(np.abs(d))[-5:].sum() / (np.abs(d).sum()+1e-9) * 100
    print(f'  {nm}: {n_changed}/{len(d)} istotnych ({n_changed*100//len(d)}%), '
          f'top-5 wymiarów = {top5_frac:.0f}% całej zmiany')

# Spójność kierunku między zdjęciami
print('\nSpójność kierunku dziurki między zdjęciami (cos):')
if len(diffs) >= 2:
    for i in range(len(diffs)):
        for j in range(i+1, len(diffs)):
            c = cos(diffs[i], diffs[j])
            print(f'  cos(diff[{i}], diff[{j}]) = {c:.4f}')
    print('  Wysokie (>0.7) = spójny kierunek = projekcja zadziała')
    print('  Niskie (<0.3)  = brak wspólnego kierunku = projekcja NIE pomoże')

# Wizualizacja rozkładu zmian
fig, axes = plt.subplots(min(len(names),3), 1, figsize=(12, 3*min(len(names),3)))
if min(len(names),3) == 1: axes = [axes]
for i in range(min(len(names),3)):
    d = diffs[i]
    thr = 0.1*np.abs(d).max()
    colors = ['red' if abs(x)>thr else 'lightgray' for x in d]
    axes[i].bar(range(len(d)), d, color=colors)
    axes[i].set_title(f'{names[i]} — zmiana per wymiar (czerwone=istotne)', fontsize=9)
    axes[i].set_xlabel('indeks wymiaru')
plt.tight_layout(); plt.show()

print('\n💡 Embedder siatkowy: zmiana SKUPIONA (lokalna, bo wymiary=komórki obrazu)')
print('   CLIP: prawdopodobnie BARDZIEJ ROZPROSZONA (abstrakcyjne cechy),')
print('   ale wciąż spójny kierunek -> dlatego projekcja, nie zerowanie wymiarów')


## 7. KROK 3 — projekcja: usuń komponent dziurki

In [ ]:
def project_out(e, direction_norm):
    """e_czyste = e - (e . v̂) v̂  — usuwa komponent wzdłuż kierunku."""
    return e - np.dot(e, direction_norm) * direction_norm

print('Wzór: e_czyste = e - (e . v̂_hole) v̂_hole\n')
for i, nm in enumerate(names):
    e = E_orig[i]
    proj_amount = np.dot(e, v_hole_norm)
    e_proj = project_out(e, v_hole_norm)
    resid = np.dot(e_proj, v_hole_norm)
    cos_before = cos(E_orig[i], E_clean[i])
    cos_after = cos(e_proj, E_clean[i])
    print(f'  {nm}:')
    print(f'     komponent dziurki w e:       {proj_amount:+.4f}')
    print(f'     po projekcji (cel ~0):       {resid:+.4f}')
    print(f'     cos(orig, clean)  PRZED:     {cos_before:.4f}')
    print(f'     cos(proj, clean)  PO:        {cos_after:.4f}  (bliżej 1.0 = lepiej)')
    print()


## 8. KROK 4 — weryfikacja: projekcja nie psuje czystych

In [ ]:
print('Czy projekcja psuje embeddingi BEZ dziurki?\n')
for i, nm in enumerate(names):
    e = E_clean[i]
    e_proj = project_out(e, v_hole_norm)
    change = np.linalg.norm(e_proj - E_clean[i])
    print(f'  {nm}: zmiana clean po projekcji = {change:.4f} (mała = projekcja chirurgiczna)')

print('\n— Podsumowanie matematyki —')
print('1. KIERUNEK:   v = mean(e_z_dziurką) - mean(e_bez)')
print('2. PROJEKCJA:  e_czyste = e - (e . v̂) v̂')
print('3. WERYFIKACJA: proj(e) . v̂ ≈ 0  oraz  clean prawie bez zmian')


## 9. Eksperymentuj sam

Pomysły do wypróbowania z własnymi zdjęciami:

- **Różne dziurki** — wgraj zdjęcia z dziurkami w różnych miejscach; zobacz, czy `v_hole` zawsze wskazuje centrum czy podąża za pozycją
- **Zdjęcia bez dziurki** — sprawdź, że detektor ich nie zaznacza (kontrola fałszywych alarmów)
- **CLIP vs fallback** — jeśli masz sieć, CLIP pokaże silniejszy efekt dziurki (||e_orig - e_clean|| większe), bo "widzi" kształt
- **Syntetyczna dziurka** — domaluj czarne koło na czystym zdjęciu (`cv2.circle(img,(x,y),r,(0,0,0),-1)`) i sprawdź, jak embedding reaguje
- **Wiele kierunków** — policz osobno kierunek jasności (ciemne vs jasne) i sprawdź, czy jest ortogonalny do v_hole

To jest fundament ogólnego pipeline'u kontroli konfoundów — następny krok projektu.
